### Получаем данные

In [4]:
from src.mlcore.dataloader import get_processed_data

# grid_resolution_ms: 100 = 100 ms, 5000 = 5 s. По умолчанию 5000. None = тиковый режим.
# Важно: ту же сетку (5000 = 5 s) используйте в run_demo, run_live (grid_resolution_ms=5000) и в run_backtest (bar_seconds=5.0).
raw = get_processed_data(
    source="raw",
    symbol="BTCUSDC",
    save_dir="parquets",
    grid_resolution_ms=60000,
)
raw

orderbook_snapshots: 100%|██████████| 2/2 [00:01<00:00,  1.35it/s]


,exchange_ts,open,high,low,close,volume,volume_1s,buy_volume_1s,sell_volume_1s,volume_imbalance_1s,...,ask_qty_90,ask_qty_91,ask_qty_92,ask_qty_93,ask_qty_94,ask_qty_95,ask_qty_96,ask_qty_97,ask_qty_98,ask_qty_99
0,2026-01-23 11:52:00,89114.500000,89130.500000,89107.398438,89118.601562,1.976,0.002,0.002,0.000,1.000000,...,0.139,0.112,0.002,0.139,1.796,0.144,0.139,0.112,0.141,0.168
1,2026-01-23 11:53:00,89151.500000,89156.203125,89120.000000,89140.000000,4.005,0.000,0.000,0.000,0.000000,...,0.138,0.133,1.010,0.219,0.114,0.007,0.135,0.033,0.133,0.112
2,2026-01-23 11:54:00,89128.398438,89154.101562,89128.398438,89142.000000,4.853,0.073,0.002,0.071,-0.945205,...,0.137,0.002,0.112,0.156,0.157,0.138,0.135,0.974,0.135,0.002
3,2026-01-23 11:55:00,89133.296875,89143.101562,89133.203125,89138.398438,0.209,0.009,0.007,0.002,0.555556,...,0.134,0.044,0.002,0.134,0.002,0.020,0.136,0.002,0.042,0.002
4,2026-01-23 11:56:00,89125.296875,89137.703125,89118.296875,89128.000000,8.170,0.000,0.000,0.000,0.000000,...,0.010,0.002,0.134,0.020,1.134,0.002,0.002,0.595,1.255,0.134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,2026-01-23 12:55:00,89180.500000,89184.703125,89180.398438,89182.796875,2.372,1.231,1.229,0.002,0.996751,...,0.042,1.749,0.414,0.102,0.012,0.001,0.104,0.104,2.020,0.116
64,2026-01-23 12:56:00,89185.000000,89199.398438,89172.101562,89196.101562,10.171,0.002,0.000,0.002,-1.000000,...,0.112,0.658,0.110,2.030,1.354,0.002,0.110,0.112,0.011,0.111
65,2026-01-23 12:57:00,89174.500000,89187.796875,89166.398438,89185.601562,3.333,0.288,0.288,0.000,1.000000,...,0.005,0.110,0.004,0.505,0.110,0.135,0.628,0.110,0.112,0.153
66,2026-01-23 12:58:00,89178.101562,89189.898438,89166.398438,89169.601562,5.797,0.189,0.145,0.044,0.534392,...,1.578,0.621,0.112,0.005,0.110,1.246,0.110,2.489,0.110,0.477


### Определяем таргеты для всех моделей

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm


def calculate_mid_target(df: pd.DataFrame):
    target = np.zeros(len(df), dtype=float)
    for i in tqdm(range(len(df) - 1)):
        current = df['close'].iloc[i]
        mid = (df['high'].iloc[i+1] + df['low'].iloc[i+1]) / 2.0
        target[i] = np.log(mid / current)
    return pd.Series(target, index=df.index)


def calculate_spread_target(df: pd.DataFrame):
    target = np.zeros(len(df), dtype=float)
    for i in tqdm(range(len(df) - 1)):
        current = df['close'].iloc[i]
        spread = df['high'].iloc[i+1] - df['low'].iloc[i+1]
        target[i] = spread / current
    return pd.Series(target, index=df.index)


mid_target = calculate_mid_target(raw)
spread_target = calculate_spread_target(raw)

### Генерируем фичи для всех моделей

In [ ]:
from src.mlcore.dataloader import calculate_features

df = calculate_features(raw, filename='all_features')
df['low'] = raw['low']
df['high'] = raw['high']
df['close'] = raw['close']
df['mid_target'] = mid_target
df['spread_target'] = spread_target
df.dropna(inplace=True)
df

In [ ]:
feature_columns = [col for col in df.columns if col not in ['mid_target', 'spread_target', 'close', 'high', 'low']]
X = df[feature_columns].copy()
y_mid = df['mid_target'].copy()
y_spread = df['spread_target'].copy()
ohlcv = df[['close', 'high', 'low']].copy()

split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_mid_train, y_mid_test = y_mid.iloc[:split_index], y_mid.iloc[split_index:]
y_spread_train, y_spread_test = y_spread.iloc[:split_index], y_spread.iloc[split_index:]
_, ohlcv_test = ohlcv.iloc[:split_index], ohlcv.iloc[split_index:]

### Обучаем mid model

In [ ]:
from src.mlcore.train import train_regression_model

mid_model = train_regression_model(
    X_train=X_train,
    y_train=y_mid_train,
    X_test=X_test,
    y_test=y_mid_test,
    n_estimators=5000,
    objective='reg:squarederror',
    model_name="model_mid",
    max_depth=3,
)

In [ ]:
from src.mlcore.evaluation import evaluate_regression_model

y_mid_pred = mid_model.predict(X_test)

evaluate_regression_model(
    y_test=y_mid_test,
    y_pred=y_mid_pred,
    length=1000,
)

### Обучаем spread model

In [ ]:
from src.mlcore.train import train_regression_model

spread_model = train_regression_model(
    X_train=X_train,
    y_train=y_spread_train,
    X_test=X_test,
    y_test=y_spread_test,
    n_estimators=5000,
    objective='reg:squaredlogerror',
    model_name="model_spread",
    max_depth=6,
)

In [ ]:
from src.mlcore.evaluation import evaluate_regression_model

y_spread_pred = spread_model.predict(X_test)

evaluate_regression_model(
    y_test=y_spread_test,
    y_pred=y_spread_pred,
    length=100000,
)